# 🧱 Part 04: Attention and Transformer Block

> **Previous context**: Part 03 built token and position vectors. Now each token needs to read information from other tokens.
> **Goal for this part**: Understand Self-Attention, Multi-Head Attention, residual connections, LayerNorm, and the Transformer Block.

Today we are solving one concrete confusion: what is the hidden mechanism behind this part of an LLM, and how can we rebuild it with small numbers before trusting a library?

## 0. The problem Attention solves

A token cannot be understood alone. In 'the bank by the river', the word 'bank' needs surrounding words to decide its meaning.

## 1. Q, K, V intuition

Query asks 'what am I looking for?', Key says 'what do I contain?', and Value is the information passed forward. Attention scores decide how much each token reads from the others.

## 2. Manual calculation

The notebook uses tiny matrices so you can compute score = QK^T, apply softmax, and multiply by V before trusting the code.

## 3. Block structure

A Transformer Block wraps Attention with residual connections, normalization, and a feed-forward network. Each part stabilizes or enriches the signal.

## How to use the code cells

Run the cells in order. The code is intentionally direct and small: each cell should expose one idea, print the key observation, and let you change a number to see what moves.

## Exercises

When a cell contains a TODO placeholder, fill it yourself and use the `assert` checks as feedback. You can ask an AI for hints, step-by-step reasoning, or a direction check, but avoid asking it to complete the exercise outright.

## Summary Checklist

- [ ] Self-Attention lets each token read other tokens.
- [ ] Multi-Head Attention reads several relationship patterns at once.
- [ ] A Transformer Block combines attention, feed-forward layers, residual paths, and normalization.

Next, continue through the code cells for the Foundation part and inspect the printed observations.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

In [ ]:
# Teaching note: follow this line to see the main step.
vocab = {"the": 0, "a": 1, "cat": 2, "dog": 3, "sat": 4,
         "ran": 5, "on": 6, "mat": 7, "[PAD]": 8, "[UNK]": 9}
vocab_size = len(vocab)
id2word = {v: k for k, v in vocab.items()}

# Teaching note: follow this line to see the main step.
sentence = ["the", "cat", "sat"]
token_ids = [vocab[w] for w in sentence]  # [0, 2, 4]

# Teaching note: follow this line to see the main step.
d_model = 4
embedding = nn.Embedding(vocab_size, d_model)
token_ids_tensor = torch.tensor(token_ids)  # [3]
X = embedding(token_ids_tensor)             # [3, 4]

print('Vocabulary')
print(f"Token IDs: {token_ids}")
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Explanation: the printed values show the main mechanism in this step.')
print(f"      X[1]='cat' → {X[1].tolist()}")
print(f"      X[2]='sat' → {X[2].tolist()}")

In [ ]:
# Teaching note: follow this line to see the main step.
seq_len = X.shape[0]  # = 3
d_k = 4

# Teaching note: follow this line to see the main step.
W_Q = nn.Linear(d_model, d_k, bias=False)
W_K = nn.Linear(d_model, d_k, bias=False)
W_V = nn.Linear(d_model, d_k, bias=False)

Q = W_Q(X)  # Teaching note: follow this line to see the main step.
K = W_K(X)  # Teaching note: follow this line to see the main step.
V = W_V(X)  # Teaching note: follow this line to see the main step.

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
# Teaching note: follow this line to see the main step.
attention_scores = Q @ K.T  # [3, 4] @ [4, 3] = [3, 3]

print('Read the values printed above and connect them to the concept in this cell.')
print(attention_scores)
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
d_k = Q.shape[-1]
scaled_scores = attention_scores / math.sqrt(d_k)

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
attention_weights = F.softmax(scaled_scores, dim=-1)

print('Read the values printed above and connect them to the concept in this cell.')
print(attention_weights)

# Teaching note: follow this line to see the main step.
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
output = attention_weights @ V  # [3, 3] @ [3, 4] = [3, 4]

print('Read the values printed above and connect them to the concept in this cell.')
print('f"\\nInput  token 0: {X[0].tolist()}"')
print('f"Output token 0: {output[0].tolist()}"')
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
def causal_mask(seq_len):
    return torch.tril(torch.ones(seq_len, seq_len))

mask = causal_mask(5)
print('Read the values printed above and connect them to the concept in this cell.')
print(mask)
# Teaching note: follow this line to see the main step.

In [ ]:
class MultiHeadAttention(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, num_heads):
        super().__init__()
        assert d_model % num_heads == 0, 'Read the values printed above and connect them to the concept in this cell.'
        
        self.d_model = d_model
        self.num_heads = num_heads
        self.d_k = d_model // num_heads  # Teaching note: follow this line to see the main step.
        
        # Teaching note: follow this line to see the main step.
        self.W_Q = nn.Linear(d_model, d_model)
        self.W_K = nn.Linear(d_model, d_model)
        self.W_V = nn.Linear(d_model, d_model)
        
        # Teaching note: follow this line to see the main step.
        self.W_O = nn.Linear(d_model, d_model)
    
    def forward(self, x, mask=None):
        '"""\n        Input: x shape = [batch, seq_len, d_model]\n        Output:   shape = [batch, seq_len, d_model]\n        """'
        batch_size, seq_len, _ = x.shape
        
        # Teaching note: follow this line to see the main step.
        #    [batch, seq_len, d_model] → [batch, num_heads, seq_len, d_k]
        Q = self.W_Q(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        # Teaching note: follow this line to see the main step.
        scores = (Q @ K.transpose(-2, -1)) / math.sqrt(self.d_k)
        
        # Teaching note: follow this line to see the main step.
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        
        # 4. Softmax
        weights = F.softmax(scores, dim=-1)
        
        # Teaching note: follow this line to see the main step.
        attn_output = weights @ V  # [batch, num_heads, seq_len, d_k]
        
        # Teaching note: follow this line to see the main step.
        attn_output = attn_output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(attn_output)


In [ ]:
# Teaching note: follow this line to see the main step.
d_model, num_heads = 8, 2
mha = MultiHeadAttention(d_model, num_heads)

test_x = torch.randn(1, 5, d_model)                         # Teaching note: follow this line to see the main step.
test_mask = causal_mask(5).unsqueeze(0).unsqueeze(0)        # [1, 1, 5, 5]

out = mha(test_x, test_mask)
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
class FeedForward(nn.Module):
    'Read the values printed above and connect them to the concept in this cell.'
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
    
    def forward(self, x):
        return self.fc2(F.relu(self.fc1(x)))


class TransformerBlock(nn.Module):
    'Decoded text'
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, num_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
    
    def forward(self, x, mask=None):
        x = self.norm1(x + self.attention(x, mask))  # Teaching note: follow this line to see the main step.
        x = self.norm2(x + self.ffn(x))              # Teaching note: follow this line to see the main step.
        return x

# Teaching note: follow this line to see the main step.
block = TransformerBlock(d_model=8, num_heads=2)
test_out = block(test_x, test_mask)
print('Read the values printed above and connect them to the concept in this cell.')

In [ ]:
# Teaching note: follow this line to see the main step.
import matplotlib.pyplot as plt

torch.manual_seed(42)

seq_len = 80
batch_size = 1
input_dim = 4
hidden_dim = 8

rnn = nn.RNN(
    input_size=input_dim,
    hidden_size=hidden_dim,
    num_layers=1,
    nonlinearity="tanh",
    batch_first=True,
)

x = torch.randn(batch_size, seq_len, input_dim, requires_grad=True)

print("=== torch.nn.RNN ===")
print(rnn)
print(f"Input shape: {tuple(x.shape)} = [batch, seq_len, input_dim]")
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
outputs, h_last = rnn(x)

print("=== Forward pass ===")
print(f"outputs shape: {tuple(outputs.shape)}")
print(f"h_last shape:  {tuple(h_last.shape)}")
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
loss = outputs[:, -1, :].pow(2).mean()
loss.backward()

# Teaching note: follow this line to see the main step.
grad_by_pos = x.grad.norm(dim=-1).squeeze(0)

print("=== Gradients from last output ===")
print(f"loss from last output: {loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    print(f"position {pos:2d} gradient norm: {grad_by_pos[pos].item():.10f}")

first_grad = grad_by_pos[0].item()
middle_grad = grad_by_pos[seq_len // 2].item()
last_grad = grad_by_pos[-1].item()

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
positions = torch.arange(seq_len)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), linewidth=2)
plt.scatter(positions.numpy(), grad_by_pos.detach().numpy(), s=10)
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN input gradient by position")
plt.grid(True, alpha=0.3)
plt.show()

near_end = grad_by_pos[-5:].mean().item()
far_start = grad_by_pos[:5].mean().item()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
def run_rnn_with_length(length):
    'Read the values printed above and connect them to the concept in this cell.'
    torch.manual_seed(42)
    model = nn.RNN(input_dim, hidden_dim, nonlinearity="tanh", batch_first=True)
    sample = torch.randn(batch_size, length, input_dim, requires_grad=True)
    out, _ = model(sample)
    loss = out[:, -1, :].pow(2).mean()
    loss.backward()
    return sample.grad.norm(dim=-1).squeeze(0).detach()

lengths = [10, 20, 40, 80]
length_results = {}

plt.figure(figsize=(7, 4))
for length in lengths:
    grads = run_rnn_with_length(length)
    length_results[length] = grads
    relative_pos = torch.linspace(0, 1, steps=length)
    plt.plot(relative_pos.numpy(), grads.numpy(), label=f"seq_len={length}")

plt.xlabel("Relative sequence position")
plt.ylabel("Input gradient norm")
plt.title("RNN gradients under different sequence lengths")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Read the values printed above and connect them to the concept in this cell.')
for length in lengths:
    grads = length_results[length]
    first = grads[0].item()
    last = grads[-1].item()
    ratio = last / max(first, 1e-30)
    print(f"  seq_len={length:2d}: first={first:.3e}, last={last:.3e}, last/first≈{ratio:.3e}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Conclusion: this small example shows the main trade-off.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
torch.manual_seed(42)

attn_dim = 8
num_heads = 2
attention = nn.MultiheadAttention(
    embed_dim=attn_dim,
    num_heads=num_heads,
    batch_first=True,
)

attn_x = torch.randn(batch_size, seq_len, attn_dim, requires_grad=True)

# Teaching note: follow this line to see the main step.
attn_mask = torch.triu(torch.ones(seq_len, seq_len, dtype=torch.bool), diagonal=1)

attn_out, attn_weights = attention(
    attn_x,
    attn_x,
    attn_x,
    attn_mask=attn_mask,
    need_weights=True,
    average_attn_weights=False,
)

attn_loss = attn_out[:, -1, :].pow(2).mean()
attn_loss.backward()

attn_grad_by_pos = attn_x.grad.norm(dim=-1).squeeze(0).detach()
last_token_weights = attn_weights[0, :, -1, :].mean(dim=0).detach()

print("=== Gradients from last Attention output ===")
print(f"loss from last output: {attn_loss.item():.6f}")
print()
for pos in [0, 1, 5, 10, 20, 40, 60, 79]:
    grad = attn_grad_by_pos[pos].item()
    weight = last_token_weights[pos].item()
    print(f"position {pos:2d}: grad={grad:.10f}, attention_weight={weight:.10f}")

print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), grad_by_pos.detach().numpy(), label="RNN")
plt.plot(positions.numpy(), attn_grad_by_pos.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Input gradient norm")
plt.title("Gradient from last position: RNN vs Attention")
plt.yscale("log")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print('Conclusion: this small example shows the main trade-off.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Reason')


In [ ]:
# Teaching note: follow this line to see the main step.
plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), last_token_weights.numpy(), linewidth=2)
plt.scatter(positions.numpy(), last_token_weights.numpy(), s=10)
plt.xlabel("Key position")
plt.ylabel("Attention weight")
plt.title("Last token attention weights")
plt.grid(True, alpha=0.3)
plt.show()

print('Conclusion: this small example shows the main trade-off.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
rnn_mass = grad_by_pos / grad_by_pos.sum()
attn_mass = attn_grad_by_pos / attn_grad_by_pos.sum()

rnn_cumsum = torch.cumsum(rnn_mass, dim=0)
attn_cumsum = torch.cumsum(attn_mass, dim=0)

plt.figure(figsize=(7, 4))
plt.plot(positions.numpy(), rnn_cumsum.numpy(), label="RNN")
plt.plot(positions.numpy(), attn_cumsum.numpy(), label="Attention")
plt.xlabel("Sequence position")
plt.ylabel("Cumulative gradient mass")
plt.title("Where does the gradient mass accumulate?")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

rnn_first_half = rnn_mass[:seq_len // 2].sum().item()
attn_first_half = attn_mass[:seq_len // 2].sum().item()

print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')
print()
print('Read the values printed above and connect them to the concept in this cell.')
print('Read the values printed above and connect them to the concept in this cell.')


In [ ]:
# Teaching note: follow this line to see the main step.
import torch

q = torch.tensor([1.0, 2.0, 0.0])
k = torch.tensor([3.0, 1.0, 4.0])

# Teaching note: follow this line to see the main step.
score = 'TODO: replace this placeholder with your code'

assert not isinstance(score, str), 'Please replace the placeholder before running the assertion.'
assert score.item() == 5.0, score
print('Exercise passed: you have understood this step.')


In [ ]:
# Teaching note: follow this line to see the main step.
import torch

seq_len = 5

# Teaching note: follow this line to see the main step.
mask = 'TODO: replace this placeholder with your code'

expected = torch.tensor([
    [1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0],
    [1, 1, 1, 0, 0],
    [1, 1, 1, 1, 0],
    [1, 1, 1, 1, 1],
])
assert not isinstance(mask, str), 'Please replace the placeholder before running the assertion.'
assert torch.equal(mask, expected), mask
print('Exercise passed: you have understood this step.')
